# Неделя 3 — Признаки и CatBoost

Задание: `docs/week3_baseline.md`

In [1]:
import sys
sys.path.append('..')
from src.features import make_features
from src.validation import evaluate



## 1. Построение признаков по группам

TODO: реализуйте make_features() в src/features.py (платежи, динамика, объём отношений, сигналы боли).

In [2]:
import pandas as pd

from src.validation import split_by_time

dataset = pd.read_parquet(
    "../data/processed/dataset.parquet"
)

train, valid, test = split_by_time(dataset)

feature_cols = [
    col for col in dataset.columns
    if col not in ["client_id", "snapshot", "target"]
]

x_train = train[feature_cols].copy()
y_train = train["target"]

x_valid = valid[feature_cols].copy()
y_valid = valid["target"]

x_test = test[feature_cols].copy()
y_test = test["target"]

print(x_train.shape, x_valid.shape, x_test.shape)

(66253, 24) (35788, 24) (40265, 24)


## 2. Обучение CatBoost

TODO: CatBoostClassifier с early stopping на eval-срезе по времени (не случайном!).

In [3]:
from catboost import CatBoostClassifier, Pool
cat_cols = [
    "segment",
    "product",
    "region"
]

for col in cat_cols:
    x_train[col] = x_train[col].fillna("unknown").astype(str)
    x_valid[col] = x_valid[col].fillna("unknown").astype(str)
    x_test[col] = x_test[col].fillna("unknown").astype(str)

numeric_cols = [
    col for col in feature_cols
    if col not in cat_cols
]

train_medians = x_train[numeric_cols].median()

x_train[numeric_cols] = x_train[numeric_cols].fillna(train_medians)
x_valid[numeric_cols] = x_valid[numeric_cols].fillna(train_medians)
x_test[numeric_cols] = x_test[numeric_cols].fillna(train_medians)


catboost_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=100
)

catboost_model.fit(
    x_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(x_valid, y_valid),
    use_best_model=True
)



0:	test: 0.8269147	best: 0.8269147 (0)	total: 81.7ms	remaining: 1m 21s
100:	test: 0.8722170	best: 0.8722170 (100)	total: 2.16s	remaining: 19.2s
200:	test: 0.8740125	best: 0.8740151 (196)	total: 4.12s	remaining: 16.4s
300:	test: 0.8748424	best: 0.8749490 (252)	total: 6.09s	remaining: 14.1s
400:	test: 0.8749668	best: 0.8750644 (324)	total: 8.06s	remaining: 12s
500:	test: 0.8749103	best: 0.8751474 (438)	total: 10.2s	remaining: 10.1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8751474047
bestIteration = 438

Shrink model to first 439 iterations.


CatBoostClassifier(depth=6, early_stopping_rounds=100, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [4]:
### Проверка на валид

catboost_valid_score = catboost_model.predict_proba(
    x_valid
)[:, 1]

catboost_valid_metrics = evaluate(
    y_valid,
    catboost_valid_score,
    "CatBoost"
)

--- CatBoost ---
ROC-AUC: 0.8751
PR-AUC: 0.6826
Precision@10%: 0.6051
Lift@10%: 7.19x
Доля оттока: 0.0842


In [5]:
### финальная проверка на тесте
catboost_test_score = catboost_model.predict_proba(
    x_test
)[:, 1]

catboost_test_metrics = evaluate(
    y_test,
    catboost_test_score,
    "CatBoost — final test"
)

--- CatBoost — final test ---
ROC-AUC: 0.9001
PR-AUC: 0.7239
Precision@10%: 0.8073
Lift@10%: 6.42x
Доля оттока: 0.1258


## 3. Итоговая таблица метрик: правило vs логрегрессия vs CatBoost



In [6]:

comparison = pd.DataFrame([
    {
        "Модель": "Правило",
        "ROC-AUC": 0.8323,
        "PR-AUC": 0.5278,
        "Precision@10%": 0.5537,
        "Lift@10%": 6.58,
        "Доля оттока": 0.0842 
        
    },
    {
        "Модель": "Логистическая регрессия",
        "ROC-AUC": 0.8532,
        "PR-AUC": 0.5780,
        "Precision@10%": 0.5539,
        "Lift@10%": 6.58,
        "Доля оттока": 0.0842
    },
    {
        "Модель": "CatBoost",
        "ROC-AUC": catboost_valid_metrics["roc_auc"],
        "PR-AUC": catboost_valid_metrics["pr_auc"],
        "Precision@10%": catboost_valid_metrics["precision_at_10"],
        "Lift@10%": catboost_valid_metrics["lift_at_10"],
        "Доля оттока": catboost_valid_metrics["churn_rate"]
    }
])

comparison.round(4)

,Модель,ROC-AUC,PR-AUC,Precision@10%,Lift@10%,Доля оттока
0,Правило,0.8323,0.5278,0.5537,6.5800,0.0842
1,Логистическая регрессия,0.8532,0.5780,0.5539,6.5800,0.0842
2,CatBoost,0.8751,0.6826,0.6051,7.1871,0.0842


## 4. Вывод на языке бизнеса

CatBoost показал лучший результат среди трех моделей.   
На финальном тесте в топ-10% клиентов по риску попадает 64% всех будущих уходов, Среди выбранных клиентов 80.7% действительно уходи, lift=6.42x